In [8]:
import pandas as pd
from moviedata.core.database import pg_connect

In [30]:
pg = pg_connect()
# ------------------------------------------------------------
# GET ALL PRICES FROM UP TO 8 WEEKS BACK

df_recent_prices = pg.sql_query(f"""
    SELECT *
    FROM medimops.product_lines
    WHERE load_timestamp >= NOW() - INTERVAL '8 weeks';
    """
)

In [51]:
# ------------------------------------------------------------
# GET AVERAGE PRICES OF FULL DF PER PRODUCT (=8 weeks)

df_recent_prices_avg = (
    df_recent_prices
    .groupby('product_link')
    .agg(
        row_count=('product_link', 'count'),
        product_title=('product_title', 'first'),
        price_8w_avg=('product_price_desktop', 'mean'),
        letterboxd_name=('letterboxd_name', 'first'),
        letterboxd_uri=('letterboxd_uri','first')
        )
)

In [50]:
# ------------------------------------------------------------
# GET LATEST PRICES OF FULL DF PER PRODUCT

df_recent_prices_latest = (
    df_recent_prices
    .sort_values(['product_link','load_timestamp'],ascending=False)
    .groupby('product_link')
    .agg({
        'product_title':'first',
        'product_price_desktop':'first'
    })
    .rename(columns={'product_price_desktop':'price_latest'})
)

In [57]:
# ------------------------------------------------------------
# JOIN DFs AND DIVIDE LATEST PRICE BY 8 WEEK AVERAGE

df_price_compare = (
    df_recent_prices_avg
    .join(df_recent_prices_latest.drop(columns=['product_title']))
)

df_price_compare['price_ratio'] = df_price_compare['price_latest']/df_price_compare['price_8w_avg']

df_price_compare = df_price_compare.sort_values('price_ratio')

In [55]:
df_low_abs_prices = (
    df_price_compare[df_price_compare['price_latest']<4]
    .sort_values('price_latest')
)

df_low_abs_prices

,row_count,product_title,price_8w_avg,letterboxd_name,letterboxd_uri,price_latest,price_ratio
product_link,,,,,,,
/adnan-g-koese-kleine-morde-blu-ray-blu-ray-M0B00AAPW94K.html,7,Kleine Morde [Blu-ray],1.790000,Shallow Grave,https://boxd.it/1XyW,1.79,1.000000
/neil-burger-the-illusionist-blu-ray-blu-ray-M0B001IXH768.html,8,The Illusionist [Blu-ray],3.470000,The Illusionist,https://boxd.it/28H6,3.47,1.000000
/len-wiseman-stirb-langsam-4-0-blu-ray-blu-ray-M0B000VXP8EE.html,1,Stirb Langsam 4.0 [Blu-ray],3.470000,Die Hard,https://boxd.it/2a7u,3.47,1.000000
/benito-zambrano-die-insel-der-zitronenblueten-blu-ray-blu-ray-M0B0BRJPG4TW.html,2,Die Insel der Zitronenblüten [Blu-ray],3.470000,The Island,https://boxd.it/28qE,3.47,1.000000
/roman-polanski-der-ghostwriter-blu-cinemathek-blu-ray-blu-ray-M0B004O4SE88.html,7,Der Ghostwriter - Blu Cinemathek [Blu-ray],3.470000,The Ghost Writer,https://boxd.it/1TQW,3.47,1.000000
/colm-mccarthy-the-girl-with-all-the-gifts-blu-ray-blu-ray-M0B01N9Z6X9L.html,7,The Girl with all the Gifts [Blu-ray],3.490000,The Gift,https://boxd.it/aJMy,3.49,1.000000
/debra-granik-winter-s-bone-blu-ray-blu-ray-M0B004L4D98Q.html,7,Winter's Bone [Blu-ray],3.490000,Winter's Bone,https://boxd.it/14FI,3.49,1.000000
/paul-king-paddington-blu-ray-blu-ray-M0B00QKCXM0C.html,7,Paddington [Blu-ray],3.490000,Paddington 2,https://boxd.it/bCF8,3.49,1.000000
/joss-whedon-avengers-age-of-ultron-blu-ray-blu-ray-M0B00WHQA1BS.html,1,Avengers - Age of Ultron [Blu-ray],3.490000,Tron,https://boxd.it/2b9w,3.49,1.000000


In [65]:
df_low_rel_prices = (
    df_price_compare[df_price_compare['row_count']>4]
    .sort_values('price_ratio')
    .head(20)
)

df_low_rel_prices

,row_count,product_title,price_8w_avg,letterboxd_name,letterboxd_uri,price_latest,price_ratio
product_link,,,,,,,
/a-very-harold-kumar-3d-christmas-blu-ray-3d-blu-ray-uv-copy-2011-region-free-blu-ray-M0B006MEVCCS.html,7,A Very Harold & Kumar 3D Christmas (Blu-ray 3D...,27.204286,Harold & Kumar Go to White Castle,https://boxd.it/1Ueu,21.49,0.789949
/ridley-scott-der-marsianer-rettet-mark-watney-3d-blu-ray-blu-ray-M0B0168YCF72.html,7,Der Marsianer - Rettet Mark Watney [3D Blu-ray],21.347143,The Martian,https://boxd.it/8SeG,17.49,0.819313
/christopher-ll-alguien-volo-sobre-el-nido-del-cuco-blu-ray-one-flew-over-the-cuckoo-s-nest-1975-import-blu-ray-M0B004UIAMW4.html,7,Alguien Voló Sobre El Nido Del Cuco (Blu-Ray) ...,40.204286,Cuckoo,https://boxd.it/wMjI,34.49,0.857869
/james-mangold-todeszug-nach-yuma-steelbook-blu-ray-blu-ray-M0B002W4MTVA.html,7,Todeszug nach Yuma (Steelbook) [Blu-ray],10.047143,3:10 to Yuma,https://boxd.it/23g0,8.79,0.874876
/rob-schrab-the-lego-movie-2-4k-ultra-hd-blu-ray-2d-blu-ray-M0B07MK45L5P.html,5,The Lego Movie 2 (4K Ultra HD) (+ Blu-ray 2D),10.810000,The Lego Movie,https://boxd.it/4pDO,9.49,0.877891
/jonathan-glazer-the-zone-of-interest-mediabook-4k-ultra-hd-blu-ray-blu-ray-M0B0CYLR7QMK.html,5,The Zone of Interest - Mediabook (4K Ultra HD)...,34.650000,The Zone of Interest,https://boxd.it/gJsA,30.49,0.879942
/the-guilty-blu-ray-blu-ray-M0B07HN55W6B.html,8,The Guilty [Blu-ray],39.090000,The Guilty,https://boxd.it/kOPW,34.49,0.882323
/jonathan-goldstein-game-night-blu-ray-fr-import-blu-ray-M0B07C5FH64G.html,7,Game night [Blu-ray] [FR Import],12.961429,Game Night,https://boxd.it/fRtu,11.59,0.894192
/andrew-niccol-bienvenue-a-gattaca-edition-deluxe-blu-ray-fr-import-blu-ray-M0B0014SM9ZQ.html,7,Bienvenue a Gattaca - Edition Deluxe [Blu-ray]...,7.790000,Gattaca,https://boxd.it/29Du,6.99,0.897304
